In [1]:
## Run this cell FIRST

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


spark = SparkSession.builder \
    .appName("Sedona") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "0") \
    .config("spark.dynamicAllocation.initialExecutors", "1") \
    .config("spark.dynamicAllocation.maxExecutors", "4") \
    .config("spark.dynamicAllocation.executorIdleTimeout", "10s") \
    .getOrCreate()

@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("EJECUTANDO:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        result.show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/23 12:30:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/23 12:30:33 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive} is set, falling back to uploading libraries under SPARK_HOME.


In [2]:
%%sql

# Run this cell SECOND


CREATE TABLE IF NOT EXISTS trips
USING DELTA
LOCATION '/spark-warehouse/trips';

CREATE TABLE IF NOT EXISTS boundary
USING DELTA
LOCATION '/spark-warehouse/boundary';

CREATE TABLE IF NOT EXISTS stations
USING DELTA
LOCATION '/spark-warehouse/stations';

CREATE TABLE IF NOT EXISTS nta
USING DELTA
LOCATION '/spark-warehouse/nta';


EJECUTANDO:
CREATE TABLE IF NOT EXISTS trips
USING DELTA
LOCATION '/spark-warehouse/trips'
++
||
++
++


EJECUTANDO:
CREATE TABLE IF NOT EXISTS boundary
USING DELTA
LOCATION '/spark-warehouse/boundary'
++
||
++
++


EJECUTANDO:
CREATE TABLE IF NOT EXISTS stations
USING DELTA
LOCATION '/spark-warehouse/stations'
++
||
++
++


EJECUTANDO:
CREATE TABLE IF NOT EXISTS nta
USING DELTA
LOCATION '/spark-warehouse/nta'
++
||
++
++



In [3]:
%%sql
describe stations;

describe trips;



EJECUTANDO:
describe stations
+---------+---------+-------+
|col_name |data_type|comment|
+---------+---------+-------+
|id       |string   |NULL   |
|name     |string   |NULL   |
|latitude |double   |NULL   |
|longitude|double   |NULL   |
|geometry |geometry |NULL   |
|boroname |string   |NULL   |
|ntacode  |string   |NULL   |
|ntaname  |string   |NULL   |
+---------+---------+-------+


EJECUTANDO:
describe trips
+------------------+---------+-------+
|col_name          |data_type|comment|
+------------------+---------+-------+
|ride_id           |string   |NULL   |
|rideable_type     |string   |NULL   |
|started_at        |timestamp|NULL   |
|ended_at          |timestamp|NULL   |
|start_station_name|string   |NULL   |
|start_station_id  |string   |NULL   |
|end_station_name  |string   |NULL   |
|end_station_id    |string   |NULL   |
|start_lat         |double   |NULL   |
|start_lng         |double   |NULL   |
|end_lat           |double   |NULL   |
|end_lng           |double   |NULL

In [4]:
%%sql

# Quey: Average trip duration and length per user type. Express the duration in hours and the length in meters.

SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_DistanceSphere(origin, target)) AS distance_meters
FROM trips
GROUP BY member_casual;

-- unidad de medida feet. pasar a metros
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:102718' ) , st_transform(target, 'EPSG:102718') )) / 3.28084  AS distance_meters
FROM trips
GROUP BY member_casual;

-- NY corresponde a UTM Zone 18N con unidad de medida en meters
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:32618' ) , st_transform(target, 'EPSG:32618') )) AS distance_meters
FROM trips
GROUP BY member_casual;



EJECUTANDO:
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_DistanceSphere(origin, target)) AS distance_meters
FROM trips
GROUP BY member_casual


26/08/23 12:31:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------------+--------------+------------------+
|member_casual|duration_hours|distance_meters   |
+-------------+--------------+------------------+
|casual       |0.36          |2123.8082035557636|
|member       |0.2           |1866.063533757892 |
+-------------+--------------+------------------+


EJECUTANDO:
-- unidad de medida feet. pasar a metros
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:102718' ) , st_transform(target, 'EPSG:102718') )) / 3.28084  AS distance_meters
FROM trips
GROUP BY member_casual


+-------------+--------------+------------------+
|member_casual|duration_hours|distance_meters   |
+-------------+--------------+------------------+
|casual       |0.36          |2124.483008083745 |
|member       |0.2           |1866.3868848791465|
+-------------+--------------+------------------+


EJECUTANDO:
-- NY corresponde a UTM Zone 18N con unidad de medida en meters
SELECT member_casual , round(AVG(duration_ms) / (60 * 60 * 1000), 2) AS duration_hours,
AVG(ST_Distance( ST_Transform(origin, 'EPSG:32618' ) , st_transform(target, 'EPSG:32618') )) AS distance_meters
FROM trips
GROUP BY member_casual


[Stage 18:=================================================>      (14 + 2) / 16]

+-------------+--------------+------------------+
|member_casual|duration_hours|distance_meters   |
+-------------+--------------+------------------+
|casual       |0.36          |2126.7376370180914|
|member       |0.2           |1866.2232066284803|
+-------------+--------------+------------------+



In [5]:
spark.sparkContext.getConf().get("spark.executor.instances", "no configurado")

'4'

In [6]:
print("APP:", spark.sparkContext.applicationId)
print("EXECUTOR MEMORY:", spark.sparkContext.getConf().get("spark.executor.memory"))
print("EXECUTOR CORES:", spark.sparkContext.getConf().get("spark.executor.cores"))
print("EXECUTOR INSTANCES:", spark.sparkContext.getConf().get("spark.executor.instances", "no configurado"))

APP: application_1787425255404_0015
EXECUTOR MEMORY: 4g
EXECUTOR CORES: 1
EXECUTOR INSTANCES: 4


In [7]:
print("Dynamic allocation:",
      spark.conf.get("spark.dynamicAllocation.enabled", "false"))

print("Shuffle tracking:",
      spark.conf.get("spark.dynamicAllocation.shuffleTracking.enabled", "false"))

print("Min executors:",
      spark.conf.get("spark.dynamicAllocation.minExecutors", "0"))

print("Initial executors:",
      spark.conf.get("spark.dynamicAllocation.initialExecutors", "1"))

print("Max executors:",
      spark.conf.get("spark.dynamicAllocation.maxExecutors", "0"))

print("Idle timeout:",
      spark.conf.get("spark.dynamicAllocation.executorIdleTimeout", "60s"))

Dynamic allocation: true
Shuffle tracking: true
Min executors: 0
Initial executors: 1
Max executors: 4
Idle timeout: 10s


In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Sedona") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "0") \
    .config("spark.dynamicAllocation.initialExecutors", "1") \
    .config("spark.dynamicAllocation.maxExecutors", "4") \
    .config("spark.dynamicAllocation.executorIdleTimeout", "60s") \
    .getOrCreate()

26/08/23 12:02:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [9]:
print(spark.conf.get("spark.dynamicAllocation.enabled"))
print(spark.conf.get("spark.dynamicAllocation.shuffleTracking.enabled"))
print(spark.conf.get("spark.dynamicAllocation.minExecutors"))
print(spark.conf.get("spark.dynamicAllocation.maxExecutors"))

true
true
0
4


In [10]:
spark.sql("""
SELECT COUNT(*)
FROM trips
""").show()

[Stage 22:====================================================>   (47 + 2) / 50]

+--------+
|count(1)|
+--------+
|35002469|
+--------+



In [6]:
%%sql

# Quey: Number of trips per picking-up borough, year, and month



SELECT boroname, EXTRACT(YEAR FROM t.started_at) AS y,
EXTRACT(MONTH FROM t.started_at) AS m, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id and boroname is not null
GROUP BY boroname, y, m
ORDER BY boroname, y, m;



EJECUTANDO:
SELECT boroname, EXTRACT(YEAR FROM t.started_at) AS y,
EXTRACT(MONTH FROM t.started_at) AS m, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id and boroname is not null
GROUP BY boroname, y, m
ORDER BY boroname, y, m


[Stage 35:====================================================>   (15 + 1) / 16]

+--------+----+---+--------+
|boroname|y   |m  |no_trips|
+--------+----+---+--------+
|Bronx   |2022|12 |9       |
|Bronx   |2023|1  |42454   |
|Bronx   |2023|2  |41188   |
|Bronx   |2023|3  |56198   |
|Bronx   |2023|4  |98481   |
|Bronx   |2023|5  |145737  |
|Bronx   |2023|6  |141793  |
|Bronx   |2023|7  |143454  |
|Bronx   |2023|8  |147539  |
|Bronx   |2023|9  |111512  |
|Bronx   |2023|10 |103005  |
|Bronx   |2023|11 |82517   |
|Bronx   |2023|12 |69120   |
|Brooklyn|2022|12 |64      |
|Brooklyn|2023|1  |409700  |
|Brooklyn|2023|2  |378585  |
|Brooklyn|2023|3  |474764  |
|Brooklyn|2023|4  |625697  |
|Brooklyn|2023|5  |786741  |
|Brooklyn|2023|6  |797724  |
+--------+----+---+--------+
only showing top 20 rows


In [30]:
%%sql

# Number of trips per starting station

SELECT s.name AS station_name, s.geometry, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id
GROUP BY station_name, geometry
ORDER BY station_name;


EJECUTANDO:
SELECT s.name AS station_name, s.geometry, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id
GROUP BY station_name, geometry
ORDER BY station_name


[Stage 215:===================================================>   (15 + 1) / 16]

+-----------------+---------------------------------------------+--------+
|station_name     |geometry                                     |no_trips|
+-----------------+---------------------------------------------+--------+
|1 Ave & E 110 St |POINT (-73.93829999999956 40.792327199857446)|23814   |
|1 Ave & E 16 St  |POINT (-73.98165557023627 40.732218530261356)|62740   |
|1 Ave & E 18 St  |POINT (-73.98054420934882 40.7338121918428)  |70994   |
|1 Ave & E 30 St  |POINT (-73.9753608200679 40.7414438700494)   |49745   |
|1 Ave & E 39 St  |POINT (-73.9711300000003 40.74714000000085)  |59256   |
|1 Ave & E 44 St  |POINT (-73.96905300997118 40.75001986039475) |32165   |
|1 Ave & E 6 St   |POINT (-73.98616880193404 40.72633085982006) |61207   |
|1 Ave & E 62 St  |POINT (-73.96094021980485 40.76122739965998) |73932   |
|1 Ave & E 68 St  |POINT (-73.95818491003202 40.76500524992444) |104722  |
|1 Ave & E 78 St  |POINT (-73.95351660023613 40.77140425984726) |13649   |
|1 Ave & E 94 St  |POINT 

In [ ]:


# Visualization: Number of trips per starting station

SELECT s.name AS station_name, s.geometry, COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id
GROUP BY station_name, geometry
ORDER BY station_name;

In [7]:
%%sql

# Quey:  Hourly number of trips that start at a station in Manhattan and end at a station located in a different borough.

SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan'
GROUP BY h
ORDER BY h;

    


EJECUTANDO:
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan'
GROUP BY h
ORDER BY h


[Stage 46:====================================================>   (15 + 1) / 16]

+---+--------+
|h  |no_trips|
+---+--------+
|0  |29007   |
|1  |17374   |
|2  |11066   |
|3  |6501    |
|4  |4831    |
|5  |4720    |
|6  |11155   |
|7  |26571   |
|8  |32322   |
|9  |28640   |
|10 |30110   |
|11 |37614   |
|12 |46448   |
|13 |54081   |
|14 |64279   |
|15 |78800   |
|16 |99005   |
|17 |128229  |
|18 |116963  |
|19 |86111   |
+---+--------+
only showing top 20 rows


In [8]:
%%sql

# Query: Monthly number of intra- and inter-borough trips per month

WITH TripsBoroughs(y, m, start_borough, end_borough, no_trips) AS (
SELECT EXTRACT(YEAR FROM t.started_at) AS y,
EXTRACT(MONTH FROM t.started_at) AS m,
s1.boroname, s2.boroname, COUNT(*)
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND
t.end_station_id = s2.id
GROUP BY y, m, s1.boroname, s2.boroname ),
IntraTrips(y, m, start_borough, no_intra_trips) AS (
SELECT y, m, start_borough, SUM(no_trips)
FROM TripsBoroughs
WHERE start_borough = end_borough
GROUP BY y, m, start_borough ),
InterTrips(y, m, start_borough, no_inter_trips) AS (
SELECT y, m, start_borough, SUM(no_trips)
FROM TripsBoroughs
WHERE start_borough != end_borough
GROUP BY y, m, start_borough )
SELECT t1.y, t1.m, t1.start_borough, t1.no_intra_trips + t2.no_inter_trips AS no_trips,
t1.no_intra_trips, t2.no_inter_trips
FROM IntraTrips t1, InterTrips t2
WHERE t1.y = t2.y AND t1.m = t2.m AND t1.start_borough = t2.start_borough
ORDER BY t1.y, t1.m, t1.start_borough;




EJECUTANDO:
WITH TripsBoroughs(y, m, start_borough, end_borough, no_trips) AS (
SELECT EXTRACT(YEAR FROM t.started_at) AS y,
EXTRACT(MONTH FROM t.started_at) AS m,
s1.boroname, s2.boroname, COUNT(*)
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND
t.end_station_id = s2.id
GROUP BY y, m, s1.boroname, s2.boroname ),
IntraTrips(y, m, start_borough, no_intra_trips) AS (
SELECT y, m, start_borough, SUM(no_trips)
FROM TripsBoroughs
WHERE start_borough = end_borough
GROUP BY y, m, start_borough ),
InterTrips(y, m, start_borough, no_inter_trips) AS (
SELECT y, m, start_borough, SUM(no_trips)
FROM TripsBoroughs
WHERE start_borough != end_borough
GROUP BY y, m, start_borough )
SELECT t1.y, t1.m, t1.start_borough, t1.no_intra_trips + t2.no_inter_trips AS no_trips,
t1.no_intra_trips, t2.no_inter_trips
FROM IntraTrips t1, InterTrips t2
WHERE t1.y = t2.y AND t1.m = t2.m AND t1.start_borough = t2.start_borough
ORDER BY t1.y, t1.m, t1.start_borough


[Stage 60:====================================================>   (15 + 1) / 16]

+----+---+-------------+--------+--------------+--------------+
|y   |m  |start_borough|no_trips|no_intra_trips|no_inter_trips|
+----+---+-------------+--------+--------------+--------------+
|2022|12 |Bronx        |9       |8             |1             |
|2022|12 |Brooklyn     |63      |52            |11            |
|2022|12 |Manhattan    |122     |113           |9             |
|2022|12 |Queens       |6       |5             |1             |
|2023|1  |Bronx        |42452   |36069         |6383          |
|2023|1  |Brooklyn     |409594  |366111        |43483         |
|2023|1  |Manhattan    |1244717 |1198278       |46439         |
|2023|1  |Queens       |94165   |75762         |18403         |
|2023|2  |Bronx        |41183   |34978         |6205          |
|2023|2  |Brooklyn     |378499  |337688        |40811         |
|2023|2  |Manhattan    |1180313 |1135380       |44933         |
|2023|2  |Queens       |91834   |73711         |18123         |
|2023|3  |Bronx        |56196   |48047  

In [9]:
%%sql

# Query: Hourly trips in weekends and weekdays.

WITH Weekday(h, no_weekday) AS (
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*)
FROM trips t, stations s
WHERE t.start_station_id = s.id AND
EXTRACT(DOW FROM t.started_at) IN (1,2,3,4,5)
GROUP BY h ),
Weekend(h, no_weekend) AS (
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*)
FROM trips t, stations s
WHERE t.start_station_id = s.id AND
EXTRACT(DOW FROM t.started_at) IN (6,0)
GROUP BY h )
SELECT w1.h, w1.no_weekday, w2.no_weekend,
w1.no_weekday + w2.no_weekend AS no_trips
FROM Weekday w1, Weekend w2
WHERE w1.h = w2.h
ORDER BY h;




EJECUTANDO:
WITH Weekday(h, no_weekday) AS (
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*)
FROM trips t, stations s
WHERE t.start_station_id = s.id AND
EXTRACT(DOW FROM t.started_at) IN (1,2,3,4,5)
GROUP BY h ),
Weekend(h, no_weekend) AS (
SELECT EXTRACT(HOUR FROM t.started_at) AS h, COUNT(*)
FROM trips t, stations s
WHERE t.start_station_id = s.id AND
EXTRACT(DOW FROM t.started_at) IN (6,0)
GROUP BY h )
SELECT w1.h, w1.no_weekday, w2.no_weekend,
w1.no_weekday + w2.no_weekend AS no_trips
FROM Weekday w1, Weekend w2
WHERE w1.h = w2.h
ORDER BY h


[Stage 79:====================================================>   (15 + 1) / 16]

+---+----------+----------+--------+
|h  |no_weekday|no_weekend|no_trips|
+---+----------+----------+--------+
|0  |304526    |72210     |376736  |
|1  |182020    |40376     |222396  |
|2  |114282    |24886     |139168  |
|3  |76896     |15738     |92634   |
|4  |75081     |15385     |90466   |
|5  |188816    |39221     |228037  |
|6  |528580    |105259    |633839  |
|7  |1107808   |213956    |1321764 |
|8  |1679895   |308258    |1988153 |
|9  |1331723   |259736    |1591459 |
|10 |1079080   |220461    |1299541 |
|11 |1144767   |238946    |1383713 |
|12 |1296010   |275579    |1571589 |
|13 |1375396   |298417    |1673813 |
|14 |1521142   |330914    |1852056 |
|15 |1673402   |366155    |2039557 |
|16 |1892524   |404832    |2297356 |
|17 |2416667   |456996    |2873663 |
|18 |2286011   |412961    |2698972 |
|19 |1700483   |322850    |2023333 |
+---+----------+----------+--------+
only showing top 20 rows


In [31]:
import json
import pydeck as pdk
import pandas as pd

# Visualization: Number of afternoon trips per destination station, starting in Manhattan and ending in another borough

df_data = spark.sql("""
SELECT s.name AS station_name, 
    ST_AsGeoJSON(geometry) AS geojson, 
    COUNT(*) AS no_trips
FROM trips t, stations s 
WHERE t.start_station_id = s.id
GROUP BY s.name, geometry
ORDER BY station_name
""").toPandas()

# Generate the quantiles
df_data['quantile_class'] = pd.qcut(df_data['no_trips'], q=8, labels=False, duplicates='drop') + 1

# Define the color palette
color_palette = {
    1: [255, 245, 245, 200],
    2: [250, 222, 211, 200],
    3: [255, 180, 203, 200],
    4: [255, 150, 211, 200],
    5: [245, 120, 103, 200],
    6: [211, 80, 50, 200],
    7: [180, 40, 30, 200],
    8: [115, 0, 0, 200]
}


features_list = [
    {
        "type": "Feature",
        "geometry": json.loads(row["geojson"]),
        "properties": {
            "station_name": str(row["station_name"]),
            "no_trips": int(row["no_trips"]),
            "fill_color": color_palette[int(row["quantile_class"])]
        }
    } for _, row in df_data.iterrows()
]

geojson_data = {"type": "FeatureCollection", "features": features_list}


layer_stations = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    stroked=True,
    filled=True,
    get_fill_color="properties.fill_color",
    point_radius_min_pixels=4,
    point_radius_scale=15,
    get_line_color=[100, 100, 100, 255],
    line_width_min_pixels=1,
    pickable=True
)

view_state = pdk.ViewState(longitude=-73.96, latitude=40.76, zoom=11)
r = pdk.Deck(
    layers=[layer_stations],
    initial_view_state=view_state,
    map_style=pdk.map_styles.LIGHT,
    tooltip={"html": "<b>Station:</b> {station_name}<br/><b>No_Trips:</b> {no_trips}"}
)

r.show()

In [32]:
%%sql

# Quey: Number of trips per starting station.


SELECT s.name AS station_name, 
        s.geometry, 
        COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id
GROUP BY station_name, geometry
ORDER BY station_name;




EJECUTANDO:
SELECT s.name AS station_name, 
        s.geometry, 
        COUNT(*) AS no_trips
FROM trips t, stations s
WHERE t.start_station_id = s.id
GROUP BY station_name, geometry
ORDER BY station_name


[Stage 236:===================================================>   (15 + 1) / 16]

+-----------------+---------------------------------------------+--------+
|station_name     |geometry                                     |no_trips|
+-----------------+---------------------------------------------+--------+
|1 Ave & E 110 St |POINT (-73.93829999999956 40.792327199857446)|23814   |
|1 Ave & E 16 St  |POINT (-73.98165557023627 40.732218530261356)|62740   |
|1 Ave & E 18 St  |POINT (-73.98054420934882 40.7338121918428)  |70994   |
|1 Ave & E 30 St  |POINT (-73.9753608200679 40.7414438700494)   |49745   |
|1 Ave & E 39 St  |POINT (-73.9711300000003 40.74714000000085)  |59256   |
|1 Ave & E 44 St  |POINT (-73.96905300997118 40.75001986039475) |32165   |
|1 Ave & E 6 St   |POINT (-73.98616880193404 40.72633085982006) |61207   |
|1 Ave & E 62 St  |POINT (-73.96094021980485 40.76122739965998) |73932   |
|1 Ave & E 68 St  |POINT (-73.95818491003202 40.76500524992444) |104722  |
|1 Ave & E 78 St  |POINT (-73.95351660023613 40.77140425984726) |13649   |
|1 Ave & E 94 St  |POINT 

In [46]:
import json
import pydeck as pdk
import pandas as pd

# Visualization: Number of trips per starting station.

df_data = spark.sql("""
SELECT s.name AS station_name, 
    ST_AsGeoJSON(geometry) AS geojson, 
    COUNT(*) AS no_trips
FROM trips t, stations s 
WHERE t.start_station_id = s.id
GROUP BY s.name, geometry
ORDER BY station_name
""").toPandas()


# Generate the quantiles
df_data['quantile_class'] = pd.qcut(df_data['no_trips'], q=8, labels=False, duplicates='drop') + 1



# Define the color palette
color_palette = {
    1: {"rgba":[255, 245, 245, 200], "name": "Off-white / Light gray"},
    2: {"rgba":[250, 222, 211, 200], "name": "Pale orange"},
    3: {"rgba":[255, 180, 203, 200], "name": "Salmon"},
    4: {"rgba":[255, 150, 211, 200], "name": "Coral"},
    5: {"rgba":[245, 120, 103, 200], "name": "Intermediate red"},
    6: {"rgba":[211, 80, 50, 200], "name": "Bright red"},
    7: {"rgba":[180, 40, 30, 200], "name": "Dark red"},
    8: {"rgba":[115, 0, 0, 200], "name": "Deep burgundy / Maroon"}
}

####
print("=== TRIPS INTERVALS ===")
intervals = pd.qcut(df_data['no_trips'], q=8, duplicates='drop')
intervals_mapping = pd.DataFrame({
    'Interval (Trip ranges)': intervals,
    'Quantile Class (Color)': df_data['quantile_class'].map(lambda x: color_palette[x]['name'])
}).drop_duplicates().sort_values(by='Interval (Trip ranges)').reset_index(drop=True)

print(intervals_mapping.to_string(index=False))
print("======================================\n")
####

features_list = [
    {
        "type": "Feature",
        "geometry": json.loads(row["geojson"]),
        "properties": {
            "station_name": str(row["station_name"]),
            "no_trips": int(row["no_trips"]),
            "fill_color": color_palette[int(row["quantile_class"])]["rgba"]
        }
    } for _, row in df_data.iterrows()
]

geojson_data = {"type": "FeatureCollection", "features": features_list}


layer_stations = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    stroked=True,
    filled=True,
    get_fill_color="properties.fill_color",
    point_radius_min_pixels=4,
    point_radius_scale=15,
    get_line_color=[100, 100, 100, 255], # Bordes grises de control para los círculos encimados
    line_width_min_pixels=1,
    pickable=True
)


view_state = pdk.ViewState(longitude=-73.96, latitude=40.76, zoom=11)
r = pdk.Deck(
    layers=[layer_stations],
    initial_view_state=view_state,
    map_style=pdk.map_styles.LIGHT,
    tooltip={"html": "<b>Station:</b> {station_name}<br/><b>No_Trips:</b> {no_trips}"}
)

r.show()

=== TRIPS INTERVALS ===
Interval (Trip ranges) Quantile Class (Color)
       (0.999, 531.25] Off-white / Light gray
      (531.25, 2238.0]            Pale orange
     (2238.0, 4043.75]                 Salmon
     (4043.75, 6768.0]                  Coral
     (6768.0, 11392.5]       Intermediate red
    (11392.5, 20382.5]             Bright red
   (20382.5, 37756.25]               Dark red
  (37756.25, 139696.0] Deep burgundy / Maroon



In [34]:
%%sql

# Quey: Number of afternoon trips per destination station, starting in Manhattan and ending in another borough

SELECT s2.name AS station_name,
       s2.geometry AS station_geometry,  
       COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan' AND
EXTRACT(HOUR FROM t.ended_at) BETWEEN 16 AND 20
GROUP BY s2.name, s2.geometry
ORDER BY s2.name


EJECUTANDO:
SELECT s2.name AS station_name,
       s2.geometry AS station_geometry,  
       COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan' AND
EXTRACT(HOUR FROM t.ended_at) BETWEEN 16 AND 20
GROUP BY s2.name, s2.geometry
ORDER BY s2.name


[Stage 260:================================================>      (14 + 2) / 16]

+----------------------+---------------------------------------------+--------+
|station_name          |station_geometry                             |no_trips|
+----------------------+---------------------------------------------+--------+
|1 Pl & Clinton St     |POINT (-73.99905708990788 40.680959099898146)|892     |
|1 St & 6 Ave          |POINT (-73.9798259999995 40.67278399999972)  |570     |
|10 St & 2 Ave         |POINT (-73.99361199999927 40.671907000000104)|181     |
|10 St & 4 Ave         |POINT (-73.9890396824147 40.66968990317174)  |210     |
|10 St & 5 Ave         |POINT (-73.9870005304379 40.6686272997222)   |303     |
|10 St & 7 Ave         |POINT (-73.98199886006942 40.66620780009703) |664     |
|100 St & Humphreys St |POINT (-73.87011999999999 40.76574000000001) |9       |
|100 St & Lewis Ave    |POINT (-73.86011999999998 40.73988000000002) |5       |
|100 St & Northern Blvd|POINT (-73.86843000000006 40.75748000000004) |7       |
|102 St & 34 Ave       |POINT (-73.86621

In [45]:
import json
import pydeck as pdk
import pandas as pd

# Visualization: Number of afternoon trips per destination station, starting in Manhattan and ending in another borough

df_data = spark.sql("""
SELECT s2.name AS station_name,
       ST_AsGeoJSON(s2.geometry) AS geojson, 
       COUNT(*) AS no_trips
FROM trips t, stations s1, stations s2
WHERE t.start_station_id = s1.id AND s1.boroname = 'Manhattan' AND
t.end_station_id = s2.id AND s2.boroname != 'Manhattan' AND
EXTRACT(HOUR FROM t.ended_at) BETWEEN 16 AND 20
GROUP BY s2.name, s2.geometry
""").toPandas()


# Generate the quantiles
df_data['quantile_class'] = pd.qcut(df_data['no_trips'], q=8, labels=False, duplicates='drop') + 1



# Define the color palette
color_palette = {
    1: {"rgba":[255, 245, 245, 200], "name": "Off-white / Light gray"},
    2: {"rgba":[250, 222, 211, 200], "name": "Pale orange"},
    3: {"rgba":[255, 180, 203, 200], "name": "Salmon"},
    4: {"rgba":[255, 150, 211, 200], "name": "Coral"},
    5: {"rgba":[245, 120, 103, 200], "name": "Intermediate red"},
    6: {"rgba":[211, 80, 50, 200], "name": "Bright red"},
    7: {"rgba":[180, 40, 30, 200], "name": "Dark red"},
    8: {"rgba":[115, 0, 0, 200], "name": "Deep burgundy / Maroon"}
}

####
print("=== TRIPS INTERVALS ===")
intervals = pd.qcut(df_data['no_trips'], q=8, duplicates='drop')
intervals_mapping = pd.DataFrame({
    'Interval (Trip ranges)': intervals,
    'Quantile Class (Color)': df_data['quantile_class'].map(lambda x: color_palette[x]['name'])
}).drop_duplicates().sort_values(by='Interval (Trip ranges)').reset_index(drop=True)

print(intervals_mapping.to_string(index=False))
print("======================================\n")
####

features_list = [
    {
        "type": "Feature",
        "geometry": json.loads(row["geojson"]),
        "properties": {
            "station_name": str(row["station_name"]),
            "no_trips": int(row["no_trips"]),
            "fill_color": color_palette[int(row["quantile_class"])]["rgba"]
        }
    } for _, row in df_data.iterrows()
]

geojson_data = {"type": "FeatureCollection", "features": features_list}


layer_stations = pdk.Layer(
    "GeoJsonLayer",
    geojson_data,
    stroked=True,
    filled=True,
    get_fill_color="properties.fill_color",
    point_radius_min_pixels=4,
    point_radius_scale=15,
    get_line_color=[100, 100, 100, 255], # Bordes grises de control para los círculos encimados
    line_width_min_pixels=1,
    pickable=True
)


view_state = pdk.ViewState(longitude=-73.96, latitude=40.76, zoom=11)
r = pdk.Deck(
    layers=[layer_stations],
    initial_view_state=view_state,
    map_style=pdk.map_styles.LIGHT,
    tooltip={"html": "<b>Station:</b> {station_name}<br/><b>No_Trips:</b> {no_trips}"}
)

r.show()




=== TRIPS INTERVALS ===
Interval (Trip ranges) Quantile Class (Color)
          (0.999, 9.0] Off-white / Light gray
           (9.0, 30.0]            Pale orange
          (30.0, 64.0]                 Salmon
         (64.0, 120.0]                  Coral
        (120.0, 200.0]       Intermediate red
        (200.0, 376.5]             Bright red
       (376.5, 718.75]               Dark red
      (718.75, 7968.0] Deep burgundy / Maroon



In [47]:
%%sql

# Query: Ratio between pick-up and returns in stations located in a radius of 500 m from Columbus circle in Manhattan, between 6 pm and 5 am.

WITH Columbus(geometry) AS (
SELECT ST_Buffer(ST_SetSRID(ST_Point(-73.981676, 40.767984), 4326), 500, true)
),
EndStationsNight(end_station_name, end_station, end_geometry, no_end_trips) AS (
SELECT t.end_station_name, t.end_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.end_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 23 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.end_station_name, t.end_station_id, s.geometry),
StartStationsNight(start_station, start_geometry, no_start_trips) AS (
SELECT t.start_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.start_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 5 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.start_station_id, s.geometry )
SELECT e.end_station_name AS station, e.end_geometry, s.no_start_trips, e.no_end_trips,
round(e.no_end_trips / s.no_start_trips::numeric, 2) AS ratio
FROM EndStationsNight e, StartStationsNight s
WHERE e.end_station = s.start_station;



EJECUTANDO:
WITH Columbus(geometry) AS (
SELECT ST_Buffer(ST_SetSRID(ST_Point(-73.981676, 40.767984), 4326), 500, true)
),
EndStationsNight(end_station_name, end_station, end_geometry, no_end_trips) AS (
SELECT t.end_station_name, t.end_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.end_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 23 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.end_station_name, t.end_station_id, s.geometry),
StartStationsNight(start_station, start_geometry, no_start_trips) AS (
SELECT t.start_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.start_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 5 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.start_station_id, s.geometry )
SELECT e.end_station_name AS station, e.end_geometry, s.no_start_trips, e.no_en

+-----------------------------+---------------------------------------------+--------------+------------+-----+
|station                      |end_geometry                                 |no_start_trips|no_end_trips|ratio|
+-----------------------------+---------------------------------------------+--------------+------------+-----+
|Broadway & W 56 St           |POINT (-73.98192337897358 40.765265398921024)|2552          |12086       |4.74 |
|Broadway & W 58 St           |POINT (-73.98169332932152 40.76695316965493) |1593          |23584       |14.80|
|W 63 St & Broadway           |POINT (-73.98261427980944 40.77163851033052) |956           |13343       |13.96|
|W 56 St & 8 Ave              |POINT (-73.98313522304855 40.76595918138116) |2400          |16696       |6.96 |
|Broadway & W 58 St           |POINT (-73.98169332930108 40.766953169641816)|1384          |2189        |1.58 |
|7 Ave & Central Park South   |POINT (-73.97906899458894 40.76674056444456) |1514          |16304       

In [49]:
import json
import pydeck as pdk
import pandas as pd

# Visualization : Ratio between pick-up and returns in stations located in a radius of 500 m from Columbus circle in Manhattan, between 6 pm and 5 am.
df_data = spark.sql("""
WITH Columbus(geometry) AS (

SELECT ST_Buffer(ST_SetSRID(ST_Point(-73.981676, 40.767984), 4326), 500,  true)
),

EndStationsNight(end_station_name, end_station, end_geometry, no_end_trips) AS (
SELECT t.end_station_name, t.end_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.end_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 23 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY end_station_name, t.end_station_id, s.geometry),

StartStationsNight(start_station, start_geometry, no_start_trips) AS (
SELECT t.start_station_id, s.geometry, COUNT(*)
FROM Columbus c, trips t, stations s
WHERE t.start_station_id = s.id AND ST_Contains(c.geometry, s.geometry) AND
(EXTRACT(HOUR FROM t.started_at) BETWEEN 18 and 5 OR
EXTRACT(HOUR FROM t.started_at) BETWEEN 0 and 5)
GROUP BY t.start_station_id, s.geometry )

SELECT e.end_station_name AS station, 
ST_AsGeoJSON(e.end_geometry) AS geojson, 
s.no_start_trips, e.no_end_trips,
round(e.no_end_trips / s.no_start_trips::numeric, 2) AS ratio
FROM EndStationsNight e, StartStationsNight s
WHERE e.end_station = s.start_station;

""").toPandas()


features_boundary = [
{
"type": "Feature",
"geometry": json.loads(row["geojson"]),
"properties": {
"station": str(row["station"]),
"ratio": float(row["ratio"])
}
}
for _, row in df_data.iterrows()
]

geojson_boundary = {"type": "FeatureCollection", "features": features_boundary}


layer_station = pdk.Layer(
"GeoJsonLayer",
geojson_boundary,
stroked=True,
filled=True,
get_fill_color=[200, 0, 80, 255],
get_line_color=[200, 0, 80, 255],
get_point_radius=30,  
line_width_min_pixels=3,
pickable=True
)




# LAyer 500 Around Columbus
df_columbus = spark.sql("""
SELECT ST_AsGeoJSON(ST_Buffer(ST_SetSRID(ST_Point(-73.981676, 40.767984), 4326), 500,  true)) as geojson
""").toPandas()

features_columbus = [
{
"type": "Feature",
"geometry": json.loads(row["geojson"])
}
for _, row in df_columbus.iterrows()
]


geojson_columbus = {"type": "FeatureCollection", "features": features_columbus}

layer_columbus = pdk.Layer(
"GeoJsonLayer",
geojson_columbus,
stroked=True,
filled=False,
get_line_color=[200, 220, 80, 255],
line_width_min_pixels=3,
pickable=False
)

# Map centered in New York
view_state = pdk.ViewState(longitude=-73.96, latitude=40.76, zoom=11)

r = pdk.Deck(
layers=[layer_station, layer_columbus],
initial_view_state=view_state,

map_style=pdk.map_styles.LIGHT,
tooltip={"html": "<b>Station:</b> {station}<br/><b>Ratio:</b> {ratio}"}
)


r.show()